In [1]:
import sqlite3

In [2]:
from datetime import datetime

In [3]:
# Initialization
def get_connection():
    return sqlite3.connect("telehealth.db")
def get_connection():
    return sqlite3.connect("telehealth.db")

# -----------------------------
# Database initialization
# -----------------------------
def init_db():
    conn = get_connection()
    conn.execute("PRAGMA foreign_keys = ON")
    cursor = conn.cursor()

    # Patients table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS patients (
        patient_id INTEGER PRIMARY KEY,
        first_name TEXT NOT NULL,
        last_name TEXT NOT NULL,
        age INTEGER,
        phone TEXT NOT NULL,
        email TEXT UNIQUE
    )
    """)

    # Appointments table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS appointments (
        appointment_id INTEGER PRIMARY KEY AUTOINCREMENT,
        patient_id INTEGER,
        date_time TEXT,
        status TEXT,
        FOREIGN KEY(patient_id) REFERENCES patients(patient_id)
            ON DELETE RESTRICT
            ON UPDATE CASCADE
    )
    """)

    # Triage table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS triage (
        triage_id INTEGER PRIMARY KEY AUTOINCREMENT,
        appointment_id INTEGER,
        patient_id INTEGER,
        visit_date TEXT,
        bp_systolic INTEGER,
        bp_diastolic INTEGER,
        pulse INTEGER,
        weight REAL,
        height REAL,
        bmi REAL,
        rbs REAL,
        fbs REAL,
        symptoms TEXT,
        danger TEXT,
        FOREIGN KEY(patient_id) REFERENCES patients(patient_id)
            ON DELETE CASCADE
            ON UPDATE CASCADE,
        FOREIGN KEY(appointment_id) REFERENCES appointments(appointment_id)
            ON DELETE CASCADE
            ON UPDATE CASCADE
    )
    """)

    # Consultation table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS consultation (
        consult_id INTEGER PRIMARY KEY AUTOINCREMENT,
        appointment_id INTEGER,
        patient_id INTEGER,
        triage_id INTEGER,
        diagnosis TEXT,
        notes TEXT,
        meds TEXT,
        plan TEXT,
        next_review TEXT,
        FOREIGN KEY(patient_id) REFERENCES patients(patient_id)
            ON DELETE CASCADE
            ON UPDATE CASCADE,
        FOREIGN KEY(appointment_id) REFERENCES appointments(appointment_id)
            ON DELETE CASCADE
            ON UPDATE CASCADE,
        FOREIGN KEY(triage_id) REFERENCES triage(triage_id)
            ON DELETE CASCADE
            ON UPDATE CASCADE
    )
    """)

    conn.commit()
    conn.close()

# Initialize database
init_db()

In [4]:
class Patient:
    def __init__(self, patient_id, first_name, last_name, age, phone, email):
        self.patient_id = patient_id
        self.first_name = first_name
        self.last_name = last_name
        self.age = age
        self.phone = phone
        self.email = email

    def save(self):
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO patients (patient_id, first_name, last_name, age, phone, email)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (self.patient_id, self.first_name, self.last_name, self.age, self.phone, self.email))
        conn.commit()
        conn.close()
        
#get all patients
    @staticmethod
    def all():
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM patients")
        rows = cursor.fetchall()
        conn.close()
        return rows

    
#get a specific patient 
    @staticmethod
    def get_by_id(patient_id):
        with sqlite3.connect("telehealth_1.db") as conn:
            conn = get_connection()
            cursor = conn.cursor()
            cursor.execute("SELECT * FROM patients WHERE patient_id = ?", (patient_id,))
            row = cursor.fetchone()
            conn.close()
        return row


In [5]:
class Appointment:
    STATUS = ["Scheduled", "Arrived", "Triaged", "Completed", "Missed"]

    def __init__(self, appointment_id, patient_id, date_time, status="Scheduled"):
        self.appointment_id = appointment_id
        self.patient_id = patient_id
        self.date_time = date_time
        self.status = status

    def save(self):
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO appointments (appointment_id, patient_id, date_time, status)
            VALUES (?, ?, ?, ?)
        """, (self.appointment_id, self.patient_id, self.date_time, self.status))
        conn.commit()
        conn.close()

    @staticmethod
    def update_status(appointment_id, status):
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute("""
            UPDATE appointments
            SET status=?
            WHERE appointment_id=?
        """, (status, appointment_id))
        conn.commit()
        conn.close()

    @staticmethod
    def all():
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM appointments ORDER BY date_time DESC")
        rows = cursor.fetchall()
        conn.close()
        return rows

    @staticmethod
    def for_patient(patient_id):
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute("""
            SELECT * FROM appointments
            WHERE patient_id=?
            ORDER BY date_time DESC
        """, (patient_id,))
        rows = cursor.fetchall()
        conn.close()
        return rows


In [ ]:
# Common symptoms with their assigned severity scores
"""COMMON_SYMPTOMS = {
    "Fever": 2,
    "Cough": 2,
    "Fatigue": 1,
    "Shortness of Breath": 4,
    "Headache": 3,
    "Nausea": 2,
    "Vomiting": 3,
    "Diarrhea": 2,
    "Chest Pain": 5
}"""


In [6]:
class Triage:
    def __init__(self, appointment_id, patient_id, bp_systolic, bp_diastolic, pulse,
                 weight, height, rbs, fbs, symptoms, danger):
        self.appointment_id = appointment_id
        self.patient_id = patient_id
        self.visit_date = str(datetime.now())
        self.bp_systolic = bp_systolic
        self.bp_diastolic = bp_diastolic
        self.pulse = pulse
        self.weight = weight
        self.height = height
        self.bmi = round(weight / ((height / 100) ** 2), 2) if height else 0
        self.rbs = rbs
        self.fbs = fbs
        self.symptoms = symptoms
        self.danger = danger

    def save(self):
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO triage (appointment_id, patient_id, visit_date,
                                bp_systolic, bp_diastolic, pulse,
                                weight, height, bmi, rbs, fbs,
                                symptoms, danger)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (self.appointment_id, self.patient_id, self.visit_date,
              self.bp_systolic, self.bp_diastolic, self.pulse,
              self.weight, self.height, self.bmi, self.rbs, self.fbs,
              self.symptoms, self.danger))
        conn.commit()
        conn.close()
        Appointment.update_status(self.appointment_id, "Triaged")

    @staticmethod
    def for_appointment(appointment_id):
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM triage WHERE appointment_id=?", (appointment_id,))
        rows = cursor.fetchall()
        conn.close()
        return rows

In [8]:
class Consultation:
    def __init__(self, appointment_id, patient_id, triage_id, diagnosis, notes, meds, plan, next_review):
        self.appointment_id = appointment_id
        self.patient_id = patient_id
        self.triage_id = triage_id
        self.diagnosis = diagnosis
        self.notes = notes
        self.meds = meds
        self.plan = plan
        self.next_review = str(next_review)

    def save(self):
        conn = get_connection()
        conn.execute("PRAGMA foreign_keys = ON")
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO consultation (appointment_id, patient_id, triage_id,
                                      diagnosis, notes, meds, plan, next_review)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (self.appointment_id, self.patient_id, self.triage_id,
              self.diagnosis, self.notes, self.meds, self.plan, self.next_review))
        
        Appointment.update_status(self.appointment_id, "Completed")

        if self.next_review:
            existing_appts = Appointment.all()
            new_id = max([a[0] for a in existing_appts] + [0]) + 1
            next_appt = Appointment(new_id, self.patient_id, self.next_review, "Scheduled")
            next_appt.save()

        conn.commit()
        conn.close()

In [ ]:
class NotificationService:
    @staticmethod
    def send_reminder(patient, appointment):
        print(f"Reminder: {patient.first_name} {patient.last_name}, "
              f"appointment at {appointment.date_time}")

In [10]:
import csv

with open("patients.csv") as file:
    reader = csv.DictReader(file)
    for row in reader:
        p = Patient(
            int(row["patient_id"]),
            row["first_name"],
            row["last_name"],
            int(row["age"]),
            row["phone"],
            row["email"]
        )
        p.save_to_db()


with open("appointments.csv") as file:
    reader = csv.DictReader(file)
    for row in reader:
        a = Appointment(
            int(row["appointment_id"]),
            int(row["patient_id"]),
            row["date_time"],
            row["status"]
        )
        a.save_to_db()

print("CSV data loaded into the database successfully!")


CSV data loaded into the database successfully!


In [ ]:
# CLI interface for testing
# Interactive Telehealth Menu
while True:
    print("\n Telehealth Management Menu ")
    print("1. Register New Patient")
    print("2. Schedule Appointment")
    print("3. View Appointments & Send Reminders")
    print("4. Run Symptom Triage")
    print("5. Exit")

    choice = input("Enter your choice (1-5): ").strip()


    if choice == "1":   # Register New Patient 

        first_name = input("First Name: ")
        last_name = input("Last Name: ")
        age = int(input("Age: "))
        phone = input("Phone: ")
        email = input("Email: ")

        existing_patients = Patient.list_all()
        existing_ids = [row[0] for row in existing_patients]
        patient_id = max(existing_ids + [0]) + 1

        new_patient = Patient(patient_id, first_name, last_name, age, phone, email)
        new_patient.save_to_db()

        print(f"Patient {first_name} {last_name} registered with ID {patient_id}")


    elif choice == "2":   # Schedule Appointment 

        patients = Patient.list_all()

        print("Available Patients:")
        for row in patients:
            print(f"{row[0]}: {row[1]} {row[2]}")

        patient_id = int(input("Enter Patient ID to schedule for: "))
        date_time = input("Appointment Date & Time (YYYY-MM-DD HH:MM): ")

        # Generate next appointment ID and save
        all_appointments = []
        for row in patients:
            all_appointments.extend(Appointment.for_patient(row[0]))

        existing_ids = [appt[0] for appt in all_appointments]
        appointment_id = max(existing_ids + [0]) + 1

        new_appointment = Appointment(appointment_id, patient_id, date_time)
        new_appointment.save_to_db()

        print(f"Appointment for Patient ID {patient_id} scheduled at {date_time}")


    elif choice == "3":   # View Appointments & Reminders 

        patients = Patient.list_all()

        for row in patients:
            current_patient = Patient(*row)

            appointments = Appointment.for_patient(current_patient.patient_id)

            for appt in appointments:
                current_appointment = Appointment(*appt)

                print(
                    f"{current_patient.first_name} {current_patient.last_name} "
                    f"- {current_appointment.date_time} - {current_appointment.status}"
                )

                send = input(f"Send reminder to {current_patient.first_name}? (y/n): ").lower()

                if send == "y":
                    NotificationService.send_reminder(current_patient, current_appointment)


    elif choice == "4":
        
        patients = Patient.list_all()

        for p_row in patients:
            print(f"{p_row[0]}: {p_row[1]} {p_row[2]}")

        patient_id = int(input("Enter Patient ID for triage: "))
        selected_patient = Patient(*[p for p in patients if p[0]==patient_id][0])

        existing_symptoms = Triage.list_symptoms()
        all_symptoms = list(set(COMMON_SYMPTOMS.keys()) | set(existing_symptoms))

        print("Available symptoms to choose from:")
        for idx, symptom in enumerate(all_symptoms, 1):
            print(f"{idx}. {symptom}")

        choices_input = input("Select symptoms by number (comma separated): ")
        selected_indices = [int(idx.strip()) - 1 for idx in choices_input.split(",")]
        selected_symptoms = [all_symptoms[i] for i in selected_indices]

        Triage.save_symptoms(patient_id, selected_symptoms)

        # Calculate total severity score
        total_score = sum(Triage.get_symptom_weight(symptom) for symptom in selected_symptoms)
        severity = Triage.assess_severity(total_score)

        print(f"Triage Severity: {severity}")
        if severity == "Severe":
            Triage.escalate_patient(selected_patient)



    elif choice == "5":
        print("Exiting Telehealth Management Menu. Goodbye!")
        break


    else:
        print("Invalid choice. Please enter a number 1-5.")




 Telehealth Management Menu 
1. Register New Patient
2. Schedule Appointment
3. View Appointments & Send Reminders
4. Run Symptom Triage
5. Exit


1: Alice Maina
2: Brian Kariuki
3: Clara Wanjiku
4: David Otieno
5: Esther Muthoni
6: Bridget Maina


AttributeError: 'list' object has no attribute 'keys'